In [ ]:
# === SatQuery AI: Pipeline A — VQA & Captioning on BigEarthNet (Sentinel-1 SAR) ===
!pip install -q -U transformers accelerate peft bitsandbytes datasets pillow huggingface_hub



In [ ]:
import os, json, torch, random
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoProcessor,
    LlavaForConditionalGeneration,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from huggingface_hub import HfApi, login

HF_TOKEN = os.environ.get('HF_TOKEN', '')
if HF_TOKEN:
    login(token=HF_TOKEN)

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))



In [ ]:
# === Load Dataset & Prepare QA Pairs ===
DATA_DIR = '/kaggle/input/bigearthnetsentinel1'
images_list = []
if os.path.exists(DATA_DIR):
    for root, _, files in os.walk(DATA_DIR):
        for f in files:
            if f.endswith(('.jpg', '.png', '.tif')):
                images_list.append(os.path.join(root, f))
    print(f'Found {len(images_list)} SAR patches.')
else:
    print('Creating synthetic sample pairs for initial fine-tuning verification...')
    os.makedirs('sample_data', exist_ok=True)
    for i in range(200):
        img = Image.new('RGB', (224, 224), color=(random.randint(20, 80), random.randint(30, 90), random.randint(40, 100)))
        p = f'sample_data/patch_{i}.jpg'
        img.save(p)
        images_list.append(p)

TEMPLATES = [
    ('What type of land cover is visible in this remote sensing SAR imagery?', 'The SAR backscatter indicates a structured mix of agricultural terrain, crop canopies, and scattered rural settlement structures.'),
    ('Describe the visual features of this satellite image.', 'This scene presents distinct radar reflectance signatures highlighting surface roughness, vegetation volume scattering, and clear boundary delineations.'),
    ('Is there water or moisture present in this area?', 'Low specular backscatter regions suggest possible surface water accumulation and moist soil conditions.')
]

dataset_samples = []
for p in images_list[:500]:
    q, a = random.choice(TEMPLATES)
    dataset_samples.append({'image_path': p, 'question': q, 'answer': a})

print(f'Generated {len(dataset_samples)} training samples.')



In [ ]:
# === Load 4-Bit Base Model ===
BASE_MODEL = 'llava-hf/llava-1.5-7b-hf'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

processor = AutoProcessor.from_pretrained(BASE_MODEL)
model = LlavaForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16
)

model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()



In [ ]:
# === Training Loop ===
class RSVQADataset(Dataset):
    def __init__(self, samples, processor):
        self.samples = samples
        self.processor = processor

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        try:
            image = Image.open(item['image_path']).convert('RGB')
        except Exception:
            image = Image.new('RGB', (224, 224), (50, 50, 50))
        
        prompt = f"USER: <image>\n{item['question']}\nASSISTANT: {item['answer']}"
        inputs = self.processor(text=prompt, images=image, return_tensors='pt', padding='max_length', max_length=128, truncation=True)
        return {k: v.squeeze(0) for k, v in inputs.items()}

train_dataset = RSVQADataset(dataset_samples, processor)

train_args = TrainingArguments(
    output_dir='./results_vqa',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    fp16=True,
    save_strategy='no',
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=train_dataset
)

print('Starting QLoRA fine-tuning...')
trainer.train()
print('Training completed!')



In [ ]:
# === Save & Push to Hugging Face Hub ===
SAVE_DIR = './satquery_ai_vqa_lora'
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

HF_REPO = 'mokshda/satquery-ai-vqa-lora'
try:
    api = HfApi()
    api.create_repo(repo_id=HF_REPO, exist_ok=True, private=False)
    api.upload_folder(
        folder_path=SAVE_DIR,
        repo_id=HF_REPO,
        repo_type='model'
    )
    print(f'Successfully uploaded adapter weights to https://huggingface.co/{HF_REPO}')
except Exception as e:
    print('Upload error (check HF_TOKEN):', e)

